In [38]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split

In [39]:
# Import data
data = pd.read_csv(r'C:\Users\jaskew\Documents\project_repository\data\processed\CDC.csv', parse_dates=['obs_date'])

# Rename 'obs_date' to 'date' and set it as the index
data.rename(columns={'obs_date': 'date'}, inplace=True)
data.set_index('date', inplace=True)

# Display the first 10 rows
data.head(10)

,indicator,API_UserName,OpDiv,observations,curr_date
date,,,,,
2025-01-01,146.71.50.198,818860012482918321,CDC,1,2025-01-01
2025-01-01,149.36.49.225,818860012482918321,CDC,28,2025-01-01
2025-01-01,162.142.125.242,818860012482918321,CDC,3,2025-01-01
2025-01-01,162.142.125.247,818860012482918321,CDC,2,2025-01-01
2025-01-01,185.230.63.171,818860012482918321,CDC,6,2025-01-01
2025-01-01,23.26.221.12,818860012482918321,CDC,49,2025-01-01
2025-01-01,23.26.221.2,818860012482918321,CDC,51,2025-01-01
2025-01-01,23.26.221.4,818860012482918321,CDC,36,2025-01-01
2025-01-01,34.160.111.145,818860012482918321,CDC,4,2025-01-01


In [40]:
def create_feature_enriched_dataset(pivot_df, history_days=14, forecast_days=7):
    X, y, indicator_names = [], [], []

    for indicator in pivot_df.columns:
        series = pivot_df[indicator].fillna(0).astype(int).values
        dates = pivot_df.index

        for i in range(history_days, len(series) - forecast_days):
            window = series[i - history_days:i]
            future_window = series[i:i + forecast_days]
            label = 1 if future_window.sum() > 0 else 0

            # Feature engineering
            days_since_seen = history_days - np.argmax(window[::-1]) if np.any(window) else history_days
            rolling_3 = window[-3:].sum()
            rolling_7 = window[-7:].sum()
            avg_14 = window.mean()
            max_14 = window.max()

            features = [
                *window,             # original raw activity (14)
                days_since_seen,    # 1
                rolling_3,          # 1
                rolling_7,          # 1
                avg_14,             # 1
                max_14              # 1
            ]

            X.append(features)
            y.append(label)
            indicator_names.append(indicator)

    return np.array(X), np.array(y), np.array(indicator_names)


In [41]:
# Convert observations to binary appearance flags
pivot_df = data.pivot_table(index='date', columns='indicator', values='observations', aggfunc='count').fillna(0)
pivot_df = (pivot_df > 0).astype(int)

# Ensure daily frequency and fill missing values
pivot_df = pivot_df.asfreq('D').fillna(0)
top_indicators = pivot_df.sum().nlargest(10).index  # Select top 10 indicators
pivot_df = pivot_df[top_indicators]


In [42]:
X, y, indicators = create_feature_enriched_dataset(pivot_df)

print(f"X shape: {X.shape}")        # [samples, 14]
print(f"y shape: {y.shape}")        # [samples]
print(f"Sample input:\n{X[0]}")
print(f"Sample label: {y[0]} (for indicator: {indicators[0]})")


X shape: (830, 19)
y shape: (830,)
Sample input:
[ 0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0. 14.  0.  0.  0.
  0.]
Sample label: 0 (for indicator: 104.21.48.1)


In [43]:
from imblearn.over_sampling import SMOTE

# === Step 1: Split the enriched dataset ===
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Apply SMOTE to oversample the minority class
smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)

# Check the new class distribution
unique, counts = np.unique(y_train, return_counts=True)
print("Balanced Class Distribution:", dict(zip(unique, counts)))

Balanced Class Distribution: {np.int64(0): np.int64(620), np.int64(1): np.int64(620)}


In [44]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 4, 5],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

grid_search = GridSearchCV(
    estimator=XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    param_grid=param_grid,
    scoring='roc_auc',
    cv=3,
    verbose=1
)
grid_search.fit(X_train, y_train)
print("Best Parameters:", grid_search.best_params_)

Fitting 3 folds for each of 72 candidates, totalling 216 fits


C:\Users\jaskew\AppData\Roaming\Python\Python311\site-packages\xgboost\training.py:183: UserWarning: [14:53:03] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\jaskew\AppData\Roaming\Python\Python311\site-packages\xgboost\training.py:183: UserWarning: [14:53:04] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\jaskew\AppData\Roaming\Python\Python311\site-packages\xgboost\training.py:183: UserWarning: [14:53:04] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\jaskew\AppData\Roaming\Python\Python311\site-packages\xgboost\training.py:183: UserWarning: [14:53:04] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\lear

Best Parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200, 'subsample': 1.0}


In [ ]:
# === Step 1: Split the enriched dataset ===
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# === Step 3: Train XGBoost ===
xgb_clf = XGBClassifier(
    n_estimators=00,
    learning_rate=0.1,
    max_depth=3,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)

xgb_clf.fit(X_train, y_train)

C:\Users\jaskew\AppData\Roaming\Python\Python311\site-packages\xgboost\training.py:183: UserWarning: [14:57:39] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=500, n_jobs=None,
              num_parallel_tree=None, ...)

In [77]:
# === Step 4: Predict and evaluate ===
y_pred = xgb_clf.predict(X_test)
y_prob = xgb_clf.predict_proba(X_test)[:, 1]

print("📊 XGBoost Classification Report:")
print(classification_report(y_test, y_pred))

print("🧮 Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

roc_auc = roc_auc_score(y_test, y_prob)
print(f"🏆 ROC AUC Score: {roc_auc:.4f}")

📊 XGBoost Classification Report:
              precision    recall  f1-score   support

           0       0.31      0.36      0.33        11
           1       0.95      0.94      0.95       155

    accuracy                           0.90       166
   macro avg       0.63      0.65      0.64       166
weighted avg       0.91      0.90      0.91       166

🧮 Confusion Matrix:
[[  4   7]
 [  9 146]]
🏆 ROC AUC Score: 0.8328
